# SHAP Interpretability Analysis 🇬🇧 [🇮🇹](06_interpretability_IT.ipynb)

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook applies **SHAP (SHapley Additive exPlanations)** to the tuned
XGBoost models from notebooks 04 and 05. SHAP values decompose each prediction
into per-feature contributions, providing both **global** (which features matter
most across all predictions) and **local** (why a specific prediction was made)
interpretability.

### Contents

1. Data loading — models + test sets for day and week approaches
2. SHAP values computation — TreeExplainer for XGBoost
3. Global feature importance — beeswarm summary plots
4. Dependence plots — top 5 features per approach
5. Individual predictions — waterfall plots (TP, FN, TN cases)
6. Feature importance comparison — SHAP vs XGBoost gain
7. Day vs week cross-comparison
8. Domain interpretation — training patterns and injury risk

### Key concepts

- **SHAP values** are based on cooperative game theory (Shapley values):
  each feature's contribution is the average marginal effect across all
  possible feature coalitions.
- **TreeExplainer** computes exact SHAP values for tree-based models in
  polynomial time — no sampling approximation needed.
- For binary classification, we focus on the **positive class** (injury=1)
  SHAP values throughout.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.interpretability.shap_analysis import (
    compare_feature_importance,
    compute_shap_values,
    get_shap_importance_dict,
    get_top_features,
    plot_shap_dependence,
    plot_shap_summary,
    plot_shap_waterfall,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Data Loading

We load the tuned XGBoost models saved in notebooks 04 and 05, along with
the preprocessed test sets. SHAP analysis is performed on the **test set**
to explain how the model generalizes to unseen athletes.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
_, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

# --- Week approach ---
week_model = load_model("week_best_model")
_, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

print("Day approach:")
print(f"  Model type: {type(day_model).__name__}")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")

print("\nWeek approach:")
print(f"  Model type: {type(week_model).__name__}")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")

---

## 2. SHAP Values Computation

TreeExplainer computes exact SHAP values for XGBoost in polynomial time.
For binary classification, the output has shape `(n_samples, n_features, 2)` —
we extract the positive class (injury=1) for all visualizations.

In [ ]:
# Compute SHAP values for both approaches
shap_values_day = compute_shap_values(day_model, X_test_day)
shap_values_week = compute_shap_values(week_model, X_test_week)

print(f"Day SHAP values shape: {shap_values_day.shape}")
print(f"Week SHAP values shape: {shap_values_week.shape}")

---

## 3. Global Feature Importance — Day Approach

The beeswarm (summary) plot shows **all features ranked by mean |SHAP value|**.
Each dot is one sample — its horizontal position shows the SHAP contribution,
and its color indicates the feature value (red = high, blue = low). This reveals
both **which features matter** and **how their values affect predictions**.

In [ ]:
# Beeswarm summary — Day approach
fig_summary_day = plot_shap_summary(
    shap_values_day, X_test_day,
    save_path=Path("interpretability/06_shap_summary_day"),
)
plt.show()
plt.close(fig_summary_day)

### Day approach — top features

The beeswarm plot above reveals which daily training features the model relies on
most to predict injury risk. Key patterns to look for:
- **Temporal recency**: do features from day 0 (today) dominate, or do earlier
  days (day 5-6, a week ago) carry more signal?
- **Feature type**: which training metrics (km, exertion, recovery, etc.) are
  most predictive?
- **Direction**: red dots (high values) pushing right indicate that **more** of
  that feature increases injury risk.

---

## 4. Global Feature Importance — Week Approach

In [ ]:
# Beeswarm summary — Week approach
fig_summary_week = plot_shap_summary(
    shap_values_week, X_test_week,
    save_path=Path("interpretability/06_shap_summary_week"),
)
plt.show()
plt.close(fig_summary_week)

### Week approach — top features

The week approach aggregates training data over 3-week windows (week 0 = current,
week 1 = previous, week 2 = two weeks ago). Key differences from the day approach:
- Features include **22 aggregated metrics per week** plus **3 relative km ratios**
- The relative ratios (acute:chronic load) are a well-known injury risk indicator
  in sports science
- Temporal signal is coarser but may capture longer-term load accumulation patterns

---

## 5. SHAP Dependence Plots — Top 5 Features

Dependence plots show **how a single feature's value affects the SHAP contribution**
across all test samples. The color indicates the most correlated interaction feature
(selected automatically by SHAP).

In [ ]:
# Dependence plots — Day approach (top 5 features)
top5_day = get_top_features(shap_values_day, n=5)
print(f"Day approach — top 5 features: {top5_day}\n")

for feat in top5_day:
    fig = plot_shap_dependence(
        shap_values_day, X_test_day, feat,
        save_path=Path(f"interpretability/06_dep_day_{feat}"),
    )
    plt.show()
    plt.close(fig)

### Day dependence plots — interpretation

The dependence plots above show the relationship between each top feature's raw
value (x-axis) and its SHAP contribution to the injury prediction (y-axis).

Key patterns to look for:
- **Non-linear thresholds**: sudden jumps in SHAP value at specific feature values
  may indicate training load thresholds beyond which injury risk increases
- **Interaction effects**: color gradients reveal which secondary features modify
  the primary feature's effect
- **Zero-value clusters**: samples at 0.0 represent rest days (sentinel -0.01
  replaced with 0.0 in preprocessing) — these form a distinct cluster

In [ ]:
# Dependence plots — Week approach (top 5 features)
top5_week = get_top_features(shap_values_week, n=5)
print(f"Week approach — top 5 features: {top5_week}\n")

for feat in top5_week:
    fig = plot_shap_dependence(
        shap_values_week, X_test_week, feat,
        save_path=Path(f"interpretability/06_dep_week_{feat}"),
    )
    plt.show()
    plt.close(fig)

---

## 6. Individual Predictions — Waterfall Plots

Waterfall plots explain **individual predictions** by showing how each feature
pushes the explained model output from the base value (population average) to
the final model score for that case. For `shap.TreeExplainer(model)` with
default settings, this is typically the model's raw output (for example,
log-odds in binary classification) rather than a probability. We select three
representative cases per approach:

- **True Positive (TP)**: correctly predicted injury — what features drove the alert?
- **False Negative (FN)**: missed injury — why did the model fail?
- **True Negative (TN)**: correctly predicted no injury — baseline comparison

In [ ]:
def select_case_indices(
    y_true: pd.Series, y_prob: np.ndarray
) -> dict[str, int]:
    """Select representative TP, FN, TN indices for waterfall plots.

    For each case type, picks the sample with the most extreme predicted
    probability to make the waterfall plot maximally informative.
    """
    y_pred = (y_prob >= 0.5).astype(int)
    y_arr = np.asarray(y_true)

    # True Positive: actual=1, predicted=1 — highest probability
    tp_mask = (y_arr == 1) & (y_pred == 1)
    # False Negative: actual=1, predicted=0 — lowest probability among positives
    fn_mask = (y_arr == 1) & (y_pred == 0)
    # True Negative: actual=0, predicted=0 — lowest probability
    tn_mask = (y_arr == 0) & (y_pred == 0)

    cases = {}
    if tp_mask.any():
        cases["TP"] = int(np.nonzero(tp_mask)[0][np.argmax(y_prob[tp_mask])])
    if fn_mask.any():
        cases["FN"] = int(np.nonzero(fn_mask)[0][np.argmin(y_prob[fn_mask])])
    if tn_mask.any():
        cases["TN"] = int(np.nonzero(tn_mask)[0][np.argmin(y_prob[tn_mask])])

    return cases


# --- Day approach: select cases ---
y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
day_cases = select_case_indices(y_test_day, y_prob_day)

print("Day approach — selected cases for waterfall plots:")
for case_type, idx in day_cases.items():
    print(f"  {case_type}: index={idx}, y_true={y_test_day.iloc[idx]}, "
          f"y_prob={y_prob_day[idx]:.4f}")

In [ ]:
# Waterfall plots — Day approach
for case_type, idx in day_cases.items():
    print(f"\n--- {case_type} case (index={idx}) ---")
    fig = plot_shap_waterfall(
        shap_values_day, index=idx,
        save_path=Path(f"interpretability/06_waterfall_day_{case_type.lower()}"),
    )
    plt.show()
    plt.close(fig)

### Day waterfall plots — interpretation

The waterfall plots above decompose three individual predictions:

- **TP (True Positive)**: the model correctly identified an injury risk. The
  features pushing the prediction above the base value reveal what training
  patterns the model associates with imminent injury.
- **FN (False Negative)**: the model missed this injury. Understanding which
  features kept the prediction low helps identify the model's blind spots —
  perhaps the injury had an unusual pattern (e.g., not preceded by high load).
- **TN (True Negative)**: a healthy baseline. Features should generally push
  the prediction below the base value, confirming the model's logic.

In [ ]:
# --- Week approach: select cases ---
y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
week_cases = select_case_indices(y_test_week, y_prob_week)

print("Week approach — selected cases for waterfall plots:")
for case_type, idx in week_cases.items():
    print(f"  {case_type}: index={idx}, y_true={y_test_week.iloc[idx]}, "
          f"y_prob={y_prob_week[idx]:.4f}")

In [ ]:
# Waterfall plots — Week approach
for case_type, idx in week_cases.items():
    print(f"\n--- {case_type} case (index={idx}) ---")
    fig = plot_shap_waterfall(
        shap_values_week, index=idx,
        save_path=Path(f"interpretability/06_waterfall_week_{case_type.lower()}"),
    )
    plt.show()
    plt.close(fig)

---

## 7. Feature Importance Comparison — SHAP vs XGBoost Gain

XGBoost provides built-in feature importance based on **gain** (the average
improvement in loss when a feature is used for splitting). SHAP importance
(mean |SHAP value|) is theoretically grounded and considers feature interactions.

Comparing the two reveals:
- **Agreement**: both methods rank the same features highly → robust importance
- **Disagreement**: a feature ranks high in gain but low in SHAP → it may split
  frequently but with small marginal effect, or interact strongly with others

In [ ]:
# --- Day approach: SHAP vs XGBoost gain ---
shap_imp_day = get_shap_importance_dict(shap_values_day)
builtin_imp_day = dict(zip(feature_cols_day, day_model.feature_importances_))

fig_cmp_day = compare_feature_importance(
    shap_imp_day, builtin_imp_day, top_n=15,
    save_path=Path("interpretability/06_importance_comparison_day"),
)
plt.show()
plt.close(fig_cmp_day)

In [ ]:
# --- Week approach: SHAP vs XGBoost gain ---
shap_imp_week = get_shap_importance_dict(shap_values_week)
builtin_imp_week = dict(zip(feature_cols_week, week_model.feature_importances_))

fig_cmp_week = compare_feature_importance(
    shap_imp_week, builtin_imp_week, top_n=15,
    save_path=Path("interpretability/06_importance_comparison_week"),
)
plt.show()
plt.close(fig_cmp_week)

---

## 8. Day vs Week Cross-Comparison

To compare feature importance across approaches (which use different feature sets),
we extract the **base feature type** by stripping the temporal prefix (`day_N_` or
`week_N_`). This reveals which training metrics are consistently important
regardless of the temporal granularity.

In [ ]:
import re


def extract_base_feature(feature_name: str) -> str:
    """Strip temporal prefix (day_N_ or week_N_) to get the base feature type."""
    return re.sub(r"^(day|week)_\d+_", "", feature_name)


def aggregate_importance_by_type(
    importance: dict[str, float],
) -> dict[str, float]:
    """Sum SHAP importance across temporal windows for each base feature type."""
    agg: dict[str, float] = {}
    for feat, val in importance.items():
        base = extract_base_feature(feat)
        agg[base] = agg.get(base, 0.0) + val
    return agg


# Aggregate by base feature type
day_by_type = aggregate_importance_by_type(shap_imp_day)
week_by_type = aggregate_importance_by_type(shap_imp_week)

# Find common feature types
common_types = sorted(set(day_by_type) & set(week_by_type))

print(f"Day unique feature types: {len(day_by_type)}")
print(f"Week unique feature types: {len(week_by_type)}")
print(f"Common feature types: {len(common_types)}")
print(f"\nCommon types: {common_types}")

In [ ]:
# Cross-comparison bar chart: day vs week by base feature type
if common_types:
    # Normalize to percentages for fair comparison (different scales)
    day_total = sum(day_by_type[t] for t in common_types)
    week_total = sum(week_by_type[t] for t in common_types)

    day_pct = {t: day_by_type[t] / day_total * 100 for t in common_types}
    week_pct = {t: week_by_type[t] / week_total * 100 for t in common_types}

    # Sort by average importance
    sorted_types = sorted(
        common_types,
        key=lambda t: (day_pct[t] + week_pct[t]) / 2,
        reverse=True,
    )

    x = np.arange(len(sorted_types))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(x + width / 2, [day_pct[t] for t in sorted_types],
            width, label="Day approach", color="#2196F3", alpha=0.8)
    ax.barh(x - width / 2, [week_pct[t] for t in sorted_types],
            width, label="Week approach", color="#FF9800", alpha=0.8)

    ax.set_yticks(x)
    ax.set_yticklabels(sorted_types)
    ax.set_xlabel("Relative SHAP importance (%)")
    ax.set_title("Feature Type Importance: Day vs Week Approach", fontweight="bold")
    ax.legend()
    ax.invert_yaxis()
    fig.tight_layout()

    save_figure(fig, "06_cross_comparison_day_week", subdir="interpretability",
                close=False)
    plt.show()
    plt.close(fig)

---

## 9. Domain Interpretation: Training Patterns and Injury Risk

### Connecting SHAP to sports science

The SHAP analysis bridges the gap between **black-box ML predictions** and
**actionable sports science insights**. Key interpretive themes:

#### Training load and injury risk

- **Total distance (km)** features likely appear among the top predictors. In
  sports science, acute spikes in training volume are a well-documented injury
  risk factor (Gabbett, 2016). SHAP can reveal whether the model learned this
  relationship — high km values on recent days/weeks pushing predictions toward
  injury.

- **High-intensity training** (km_z3_4, km_z5_t1_t2, km_sprinting) captures
  the quality of training load, not just volume. These metrics may interact
  with total_km — the same volume at higher intensity could carry more risk.

#### Recovery and subjective markers

- **Perceived recovery** and **perceived exertion** are subjective athlete
  reports. If these rank highly in SHAP importance, it validates the use of
  athlete self-reporting as a monitoring tool. Low perceived recovery combined
  with high training load is a classic pre-injury pattern.

- **Perceived training success** may capture athlete confidence — lower values
  could indicate early signs of overreaching (poor performance in training).

#### Temporal patterns

- **Day approach**: if features from day 0-1 (most recent) dominate over day 5-6,
  the model focuses on acute load. If earlier days also matter, the model captures
  cumulative fatigue.

- **Week approach**: the relative km ratios directly quantify acute-to-chronic
  workload ratio. Values > 1.0 indicate a training spike relative to the
  baseline — a known injury trigger in the literature.

#### Comparison with Lovdal et al. (2021)

The original paper reported that training volume features and subjective markers
were the most important predictors. Our SHAP analysis can confirm (or challenge)
these findings with a more rigorous feature attribution method.

### Limitations

- SHAP explains the **model**, not the underlying causal mechanism. High SHAP
  importance for a feature means the model uses it, not that it causes injury.
- The sentinel value replacement (ADR-007: -0.01 → 0.0) means rest days have
  zero values for all training features — the model may learn to use "zero
  clusters" as a signal, which is a valid but indirect pattern.
- TreeExplainer provides exact SHAP values for the XGBoost model but does not
  account for out-of-distribution inputs or model uncertainty.

---

## 10. Summary

### Pipeline flow

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → compute_shap_values() [TreeExplainer]
  → Global: beeswarm summary plots
  → Per-feature: dependence plots (top 5)
  → Individual: waterfall plots (TP, FN, TN)
  → Comparison: SHAP vs XGBoost gain
  → Cross-comparison: day vs week by feature type
  → All figures saved to reports/figures/interpretability/
```

### Key findings

1. **Global importance**: the beeswarm plots identify which training features
   the XGBoost models rely on most for injury prediction
2. **Dependence plots**: reveal non-linear relationships and interaction effects
   between training load features and injury risk
3. **Waterfall plots**: decompose individual predictions, showing why the model
   flagged (or missed) specific injury cases
4. **SHAP vs gain**: validates that SHAP-based and built-in importance rankings
   are broadly consistent (or highlights meaningful differences)
5. **Day vs week**: identifies which training metrics are consistently important
   across temporal granularities

### Figures generated

All figures saved to `reports/figures/interpretability/`:
- 2 beeswarm summary plots (day + week)
- 10 dependence plots (5 per approach)
- 6 waterfall plots (3 per approach: TP, FN, TN)
- 2 SHAP-vs-gain comparison charts
- 1 cross-comparison chart (day vs week)

### Next steps

- **Notebook 07**: Fairness analysis — per-group metrics using proxy groups
  (volume, injury history, data density)
- **Notebook 08**: Comparison summary — day vs week side-by-side, paper
  benchmark comparison, final recommendation